In [14]:
import pandas as pd
from pathlib import Path

ROOT_DIR = Path().cwd().parent
df = pd.read_csv(ROOT_DIR / "data/processed/diamonds_clean.csv")

In [15]:
y = df["price"]
x = df.drop(columns="price")

In [16]:
numeric_features = x.select_dtypes(include="number")

In [17]:
ordinal_features = [
    "cut",
    "color",
    "clarity"
]

ordinal_categories = [
    ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    ["J", "I", "H", "G", "F", "E", "D"],
    ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]
]

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OrdinalEncoder
)

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import (
    KFold,
    cross_validate
)

from sklearn.base import clone
import pandas as pd
import numpy as np

In [20]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [21]:
ordinal_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OrdinalEncoder(
            categories=ordinal_categories,
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
    )
])

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("ord", ordinal_pipeline, ordinal_features),
    ],
    remainder="drop"
)

In [23]:
models = {
    "LinearRegression": LinearRegression(),

    "Ridge": Ridge(
        alpha=1.0
    ),

    "RandomForest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

In [24]:
def create_pipeline(model):
    return Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("regressor", clone(model))
    ])

In [25]:
pipelines = {
    name: create_pipeline(model)
    for name, model in models.items()
}

In [26]:
print(pipelines)

{'LinearRegression': Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                         carat  depth  table     x     y     z
0       0.23   61.5   55.0  3.95  3.98  2.43
1       0.21   59.8   61.0  3.89  3.84  2.31
2       0.23   56.9   65.0  4.05  4.07  2.31
3       0.29   62.4   58.0  4.20  4.23  2.63
4       0.31   63.3   58.0  4.34  4.35  2.75
...      ...    ...    ...   ...   ...   ...
49995   0.72   60.8   57.0  5.75  5.7...
                                                  Pipeline(steps=[('imputer',
                                                            

In [27]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [28]:
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}